### 1. Agent 설치하기
pip install openai-agents

In [1]:
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)

False

### 2. Agent 만들기

In [2]:
agent = Agent(
    name="비서",
    instructions="당신은 친절하고 간결하게 답하는 비서입니다.",
)

# 주피터 노트북에는 이미 이벤트 루프가 돌고 있으므로
# run_sync() 가 아니라 async 버전인 run() 을 await 해야 합니다.
result = await Runner.run(agent, "파이썬에서 리스트와 튜플의 차이는?")
print(result.final_output)

파이썬에서 **리스트(list)**와 **튜플(tuple)**의 주요 차이는 다음과 같습니다.

| 구분 | 리스트 | 튜플 |
|---|---|---|
| 문법 | `[1, 2, 3]` | `(1, 2, 3)` |
| 수정 가능 여부 | 수정 가능 | 수정 불가 |
| 속도 | 비교적 느림 | 비교적 빠름 |
| 용도 | 변경될 데이터 | 변경되지 않을 데이터 |

```python
my_list = [1, 2, 3]
my_list[0] = 10  # 가능

my_tuple = (1, 2, 3)
# my_tuple[0] = 10  # 오류
```

튜플은 변경할 수 없으므로 딕셔너리의 키로 사용할 수 있지만, 리스트는 사용할 수 없습니다.


### Tool 추가하기

In [3]:
@function_tool
def get_weather(city: str) -> str:
    """도시의 현재 날씨를 반환합니다."""
    # 실제로는 날씨 API 호출
    return f"{city}는 현재 맑음, 22도입니다."

agent = Agent(
    name="날씨 비서",
    instructions="날씨를 물어보면 반드시 도구를 사용해 답하세요.",
    tools=[get_weather],
)

result = await Runner.run(agent, "서울 날씨 어때?")
print(result.final_output)

서울은 현재 맑고, 기온은 22도입니다.


##### 실제로 날씨를 알려주는 tool 만들기
위의 dummy 툴은 언제나 "맑음, 22도"를 표시합니다. 이제 **실제로 웹 API를 호출**하는 툴을 만들어 보겠습니다.

- 사용 API: [Open-Meteo](https://open-meteo.com/) — **API 키가 필요 없습니다**
- 두 단계로 동작합니다: `도시 이름 → 위도/경도` (지오코딩) → `위도/경도 → 현재 날씨`

**주의:** Open-Meteo의 도시 검색은 한글 이름을 잘 못 찾습니다.
(`서울` → 검색 실패, `대구` → 북한의 엉뚱한 지역을 반환)
그래서 도구에는 **영어 도시명**을 넘기고, 한글→영어 변환은 에이전트에게 맡깁니다.

In [4]:
#  https://open-meteo.com/ (Free weather forecast and historical weather API )
#  Open-Meteo가 돌려주는 날씨 코드(WMO)를 한글로 바꾸기 위한 표
WEATHER_CODE = {
    0: "맑음", 1: "대체로 맑음", 2: "부분 흐림", 3: "흐림",
    45: "안개", 48: "서리 안개",
    51: "약한 이슬비", 53: "이슬비", 55: "강한 이슬비",
    61: "약한 비", 63: "비", 65: "강한 비",
    71: "약한 눈", 73: "눈", 75: "강한 눈",
    80: "소나기", 81: "강한 소나기", 82: "매우 강한 소나기",
    95: "뇌우", 96: "우박 동반 뇌우", 99: "강한 우박 동반 뇌우",
}


def get_current_weather(city: str) -> str:
    """지정한 도시의 현재 날씨를 조회합니다.

    Args:
        city: 도시 이름을 영어로 입력 (예: Seoul, Busan, Tokyo)
    """
    # 1단계: 도시 이름 -> 위도/경도
    geo = requests.get(
        "https://geocoding-api.open-meteo.com/v1/search",
        params={"name": city, "count": 1, "language": "ko", "format": "json"},
        timeout=10,
    ).json()

    if not geo.get("results"):
        return f"'{city}' 도시를 찾을 수 없습니다. 영어 이름으로 다시 시도하세요."

    place = geo["results"][0]

    # 2단계: 위도/경도 -> 현재 날씨
    data = requests.get(
        "https://api.open-meteo.com/v1/forecast",
        params={
            "latitude": place["latitude"],
            "longitude": place["longitude"],
            "current": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
            "timezone": "auto",
        },
        timeout=10,
    ).json()

    now = data["current"]
    return (
        f"{place['name']}({place['country']}) 현재 날씨: "
        f"{WEATHER_CODE.get(now['weather_code'], '알 수 없음')}, "
        f"기온 {now['temperature_2m']}°C, "
        f"습도 {now['relative_humidity_2m']}%, "
        f"풍속 {now['wind_speed_10m']}m/s"
    )

In [5]:
# 에이전트에 붙이기 전에, 함수만 따로 호출해서 API가 잘 동작하는지 먼저 확인합니다.
print(get_current_weather("Seoul"))
print(get_current_weather("Busan"))
print(get_current_weather("NoSuchCity12345"))

서울특별시(대한민국) 현재 날씨: 대체로 맑음, 기온 19.9°C, 습도 61%, 풍속 11.2m/s


부산광역시(대한민국) 현재 날씨: 맑음, 기온 19.6°C, 습도 53%, 풍속 7.9m/s


'NoSuchCity12345' 도시를 찾을 수 없습니다. 영어 이름으로 다시 시도하세요.


In [6]:
# @function_tool 데코레이터 대신 함수로 감싸면,
# 원래 함수(get_current_weather)를 위 셀처럼 직접 테스트할 수 있어 디버깅이 편합니다.
weather_tool = function_tool(get_current_weather)

weather_agent = Agent(
    name="날씨 비서",
    instructions=(
        "당신은 친절하고 간결하게 답하는 비서입니다. "
        "날씨 질문에는 반드시 get_current_weather 도구를 사용하세요. "
        "도구에 전달하는 도시 이름은 반드시 영어로 변환해서 넣으세요(예: 서울 -> Seoul)."
    ),
    tools=[weather_tool],
)

result = await Runner.run(weather_agent, "서울, 부산, 대구 날씨 알려줘")
print(result.final_output)

현재 날씨입니다.

- 서울: 대체로 맑음, 19.9°C, 습도 61%
- 부산: 맑음, 19.6°C, 습도 53%
- 대구: 맑음, 20.8°C, 습도 51%


In [7]:
weather_tool

FunctionTool(name='get_current_weather', description='지정한 도시의 현재 날씨를 조회합니다.', params_json_schema={'properties': {'city': {'description': '도시 이름을 영어로 입력 (예: Seoul, Busan, Tokyo)', 'title': 'City', 'type': 'string'}}, 'required': ['city'], 'title': 'get_current_weather_args', 'type': 'object', 'additionalProperties': False}, on_invoke_tool=<agents.tool._FailureHandlingFunctionToolInvoker object at 0x00000276F37C3320>, strict_json_schema=True, is_enabled=True, tool_input_guardrails=None, tool_output_guardrails=None, needs_approval=False, timeout_seconds=None, timeout_behavior='error_as_result', timeout_error_function=None, defer_loading=False, custom_data_extractor=None, allowed_callers=None, output_json_schema=None)

In [8]:
weather_tool.description

'지정한 도시의 현재 날씨를 조회합니다.'

In [9]:
with trace("광주 날씨 문의"):
    result = await Runner.run(weather_agent, "광주 날씨 알려줘")

###### Now go and look at the trace

https://platform.openai.com/traces

### 핸드오프(Handoff)로 여러 에이전트 연결하기

In [10]:
billing_agent = Agent(
    name="결제 담당",
    instructions="결제 관련 문의만 처리합니다.",
)

tech_agent = Agent(
    name="기술 지원 담당",
    instructions="기술적인 문제만 처리합니다.",
)

triage_agent = Agent(
    name="상담 접수",
    instructions="문의 내용을 보고 결제 담당 또는 기술 지원 담당에게 넘기세요.",
    handoffs=[billing_agent, tech_agent],
)

result = await Runner.run(triage_agent, "제 카드에서 중복 결제됐어요")
print(result.final_output)

Tool name 'transfer_to_결제 담당' contains invalid characters for function calling and has been transformed to 'transfer_to______'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


Tool name 'transfer_to_기술 지원 담당' contains invalid characters for function calling and has been transformed to 'transfer_to_________'. Please use only letters, digits, and underscores to avoid potential naming conflicts.


중복 결제된 건을 확인하려면 결제일, 금액, 가맹점명(또는 주문번호)을 알려주세요. 카드번호 전체는 보내지 마세요.


## Guardrails 추가하기

In [11]:
from agents import Agent, Runner, GuardrailFunctionOutput, input_guardrail

BANNED_WORDS = ["씨발", "개새끼", "병신"]  # 실제로는 더 정교한 리스트/정규식 사용

@input_guardrail
def keyword_profanity_guardrail(ctx, agent, input_text) -> GuardrailFunctionOutput:
    is_offensive = any(word in input_text for word in BANNED_WORDS)
    return GuardrailFunctionOutput(
        output_info={"matched_keyword": is_offensive},
        tripwire_triggered=is_offensive,
    )

agent = Agent(
    name="비서",
    instructions="정중하게 답하세요.",
    input_guardrails=[keyword_profanity_guardrail],
)

In [12]:
result = await Runner.run(agent, "안녕하세요")
print(result.final_output)

안녕하세요! 무엇을 도와드릴까요?


In [13]:
result = await Runner.run(agent, "병신")
print(result.final_output)

InputGuardrailTripwireTriggered: Guardrail InputGuardrail triggered tripwire

### 세션(Sessions)으로 대화 이력 유지하기

In [14]:
session = SQLiteSession("user_123")  # 대화 이력이 자동 저장됨

agent = Agent(name="비서", instructions="이전 대화를 기억하고 답하세요.")

result1 = await Runner.run(agent, "제 이름은 민수예요", session=session)
result2 = await Runner.run(agent, "제 이름이 뭐라고 했죠?", session=session)
print(result2.final_output)  

민수님이라고 하셨어요.


#### 수동으로 대화 이력 관리하기 (to_input_list() 방식)

In [15]:
agent = Agent(
    name="비서",
    instructions="간결하게 답하세요.",
)

# 1턴: 첫 질문
result = await Runner.run(agent, "금문교는 어느 도시에 있어?")
print(result.final_output)  # "샌프란시스코"

# 대화 이력을 리스트(list of dicts)로 직접 추출
history = result.to_input_list()
print(history)
# [
#   {"role": "user", "content": "금문교는 어느 도시에 있어?"},
#   {"role": "assistant", "content": "샌프란시스코"}
# ]

# 2턴: 새 질문을 이전 이력 뒤에 직접 이어붙여서 전달
history.append({"role": "user", "content": "그 도시는 어느 주에 있어?"})

result = await Runner.run(agent, history)  # 리스트 전체를 입력으로 넘김
print(result.final_output)  # "캘리포니아"

# 3턴을 위해 다시 이력을 갱신
history = result.to_input_list()
print(history)

금문교는 미국 캘리포니아주 샌프란시스코에 있습니다.
[{'content': '금문교는 어느 도시에 있어?', 'role': 'user'}, {'id': 'msg_064a97e731b74f1a006ad0568733e487d08f5ea16d9cedccb3', 'content': [{'annotations': [], 'text': '금문교는 미국 캘리포니아주 샌프란시스코에 있습니다.', 'type': 'output_text', 'logprobs': []}], 'role': 'assistant', 'status': 'completed', 'type': 'message', 'phase': 'final_answer'}]


샌프란시스코는 미국 캘리포니아주에 있습니다.
[{'content': '금문교는 어느 도시에 있어?', 'role': 'user'}, {'id': 'msg_064a97e731b74f1a006ad0568733e487d08f5ea16d9cedccb3', 'content': [{'annotations': [], 'text': '금문교는 미국 캘리포니아주 샌프란시스코에 있습니다.', 'type': 'output_text', 'logprobs': []}], 'role': 'assistant', 'status': 'completed', 'type': 'message', 'phase': 'final_answer'}, {'role': 'user', 'content': '그 도시는 어느 주에 있어?'}, {'id': 'msg_064a97e731b74f1a006ad05688ae2c87d0bd545f757c45d79c', 'content': [{'annotations': [], 'text': '샌프란시스코는 미국 캘리포니아주에 있습니다.', 'type': 'output_text', 'logprobs': []}], 'role': 'assistant', 'status': 'completed', 'type': 'message', 'phase': 'final_answer'}]


In [16]:
history = []

async def chat(user_input):
    global history
    history.append({"role": "user", "content": user_input})
    result = await Runner.run(agent, history)
    history = result.to_input_list()  # 다음 턴을 위해 갱신
    return result.final_output
    #return history

print(chat("제 이름은 민수예요"))
print(chat("제 이름이 뭐였죠?"))

<coroutine object chat at 0x00000276F51C04A0>
<coroutine object chat at 0x00000276F51C04A0>


C:\Users\wins\AppData\Local\Temp\ipykernel_9616\450760959.py:11: RuntimeWarning: coroutine 'chat' was never awaited
  print(chat("제 이름은 민수예요"))
C:\Users\wins\AppData\Local\Temp\ipykernel_9616\450760959.py:12: RuntimeWarning: coroutine 'chat' was never awaited
  print(chat("제 이름이 뭐였죠?"))
